# Packaging data reconciliation | Justin Otterson
USD MSBA capstone · Bumble Bee Seafoods · 2024

**Decision:** Can transformed packaging data be trusted for sustainability reporting?

This cleanup consolidates the original validation notebooks into one explicit comparison workflow. Original saved outputs contained stale variables and mismatched categories, so no historic pass rate or business totals are repeated here. Source datasets are not bundled. This copy has not been rerun on the original data.

## Run instructions
Use a Python 3 Colab runtime with pandas, numpy and matplotlib. Upload authorized historic and melted CSV files; set filenames below. Confirm identical reporting period, unit, plant scope and packaging definitions before comparing. Configure packaging columns explicitly: never sum SKU identifiers, years, sales units or unrelated numeric fields. Category mappings require source-owner confirmation; unmatched categories remain exceptions.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

HISTORIC_FILE = "CM_Historic.csv"
MELTED_FILE = "CM_melted.csv"
# Replace with confirmed packaging quantity columns from the historic file.
PACKAGING_COLUMNS = []
TYPE_COLUMN = "Packaging Type"
AMOUNT_COLUMN = "Packaging Amount"
ABS_TOLERANCE = 1e-6  # In the confirmed source unit; adjust with domain justification.
REL_TOLERANCE = 1e-9


In [ ]:
def read_csv_checked(filename):
    if not Path(filename).exists():
        raise FileNotFoundError(f"Upload {filename} before running this section.")
    df = pd.read_csv(filename)  # Malformed rows raise; no silent skipping.
    df.columns = df.columns.map(str).str.strip()
    if df.columns.duplicated().any():
        raise ValueError("Duplicate column names after trimming.")
    return df

def numeric_checked(series, name):
    values = pd.to_numeric(series, errors="coerce")
    if values.isna().any() or not np.isfinite(values).all():
        raise ValueError(f"{name}: missing, nonnumeric or infinite quantities; resolve before comparing.")
    return values

def reconcile(historic, melted, packaging_columns, type_column, amount_column):
    if not packaging_columns or len(set(packaging_columns)) != len(packaging_columns):
        raise ValueError("Specify a unique, confirmed list of packaging quantity columns.")
    missing = set(packaging_columns) - set(historic.columns)
    if missing or not {type_column, amount_column}.issubset(melted.columns):
        raise ValueError(f"Missing historic columns: {sorted(missing)}; confirm melted schema.")
    quantities = historic[packaging_columns].apply(lambda s: numeric_checked(s, s.name))
    long = melted[[type_column, amount_column]].copy()
    if long[type_column].isna().any():
        raise ValueError("Missing packaging categories.")
    long[type_column] = long[type_column].astype(str).str.strip()
    if long[type_column].eq("").any():
        raise ValueError("Blank packaging categories.")
    long[amount_column] = numeric_checked(long[amount_column], amount_column)
    report = pd.concat([quantities.sum().rename("Historic"),
                        long.groupby(type_column)[amount_column].sum().rename("Transformed")], axis=1)
    report["Difference"] = report["Transformed"] - report["Historic"]
    both = report[["Historic", "Transformed"]].notna().all(axis=1)
    matches = np.isclose(report["Historic"], report["Transformed"], atol=ABS_TOLERANCE, rtol=REL_TOLERANCE)
    report["Status"] = np.where(~both, "Unmatched category", np.where(matches, "Within tolerance", "Investigate"))
    return report


In [ ]:
historic = read_csv_checked(HISTORIC_FILE)
melted = read_csv_checked(MELTED_FILE)
report = reconcile(historic, melted, PACKAGING_COLUMNS, TYPE_COLUMN, AMOUNT_COLUMN)
display(report)
report.to_csv("packaging_reconciliation.csv", index_label="Packaging Type")
report[["Historic", "Transformed"]].plot.bar(figsize=(12, 5), title="Packaging reconciliation — confirmed common unit")
plt.ylabel("Quantity in confirmed source unit")
plt.tight_layout()


## Interpretation and next decision
Review every unmatched category and material difference before feeding dashboard KPIs. Small floating-point residuals may be within the agreed tolerance; missing categories are never assumed to be zero. Equal aggregate totals do not prove row-level correctness, unit consistency or absence of offsetting errors. Follow with key-level reconciliation and documented category mappings. Repeat using separate inputs for each plant, preventing stale variables from contaminating comparisons.

**Contribution boundary:** The original capstone supplied the business context and validation concept. This portfolio copy includes AI-assisted refactoring and documentation in October 2026. It demonstrates the method; original-data results still require reproduction.